In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow import keras
from aijack.attack.membership import ShadowMembershipInferenceAttack
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from aijack.attack.membership import ShadowMembershipInferenceAttack


In [ ]:
vectorizer = joblib.load("vectorizer.pkl")

raw_X_train = pd.read_csv("X_train.csv")
raw_X_test  = pd.read_csv("X_test.csv")

X_train = vectorizer.transform(raw_X_train.iloc[:, 0]).toarray().astype(np.float32)
X_test  = vectorizer.transform(raw_X_test.iloc[:, 0]).toarray().astype(np.float32)

y_train = pd.read_csv("y_train.csv").values.ravel()
y_test  = pd.read_csv("y_test.csv").values.ravel()

model = load_model("model.h5")

# Model didn't work, so I had to recompile it.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Before attack 
train_acc = model.evaluate(X_train, y_train, verbose=0)[1]
test_acc  = model.evaluate(X_test,  y_test,  verbose=0)[1]

print("Before Attack")
print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test  Accuracy : {test_acc:.4f}")

In [ ]:
# AIJack doesn't use keras models directly, So a wrapper is needed.
class KerasClassifierWrapper:
    def __init__(self, model):
        self.model = model

    def predict_proba(self, X):
        probs = self.model.predict(X, verbose=0)
        return np.hstack([1 - probs, probs])

wrapped_model = KerasClassifierWrapper(model)

# 2 shadow models
shadow_models = [LogisticRegression(max_iter=1000) for _ in range(2)]
# 10 attack models
attack_models = [LogisticRegression(max_iter=1000) for _ in range(10)]

# Performs the attack on the victim (wrapped IMDB model) 
# then uses shadow models to train the attack
# and attack models to perform membership inference.
attacker = ShadowMembershipInferenceAttack(
    wrapped_model,
    shadow_models,
    attack_models
)

attacker.fit(X_train, y_train)


In [ ]:
# Membership Inference Results
in_result = attacker.predict(model.predict_proba(X_train), y_train)
out_result = attacker.predict(model.predict_proba(X_test), y_test)

# Create labels for the inferences
# '1' indicates that the instance was predicted to be in the training set (positive class)
in_label = np.ones(in_result.shape[0])

# Create labels for the inferences
# '0' indicates that the instance was predicted to be in the test set (negative class)
out_label = np.zeros(out_result.shape[0])

# Compute the accuracy score of the membership inference attack
# by concatenating the labels and results from both inferences
attack_accuracy = accuracy_score(
    np.concatenate([in_label, out_label]),  # Concatenated ground truth labels
    np.concatenate([in_result, out_result])  # Concatenated predicted labels
)

# Print the accuracy score of the membership inference attack
print("Membership Inference Attack Accuracy:", attack_accuracy)
